In [10]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_groq  import ChatGroq
from langchain.chat_models import init_chat_model


model=init_chat_model(
    "qwen/qwen3.8-27b",
    model_provider="groq",
    temperature=0,
    max_tokens=512
    
    
)




In [12]:
llm =ChatGroq(model="qwen/qwen3.8-27b")

In [13]:
llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'Qwen3.8 27B', 'status': 'beta', 'release_date': '2026-08-14', 'last_updated': '2026-08-14', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000015439572270>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000154395727B0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [14]:
import os 
from typing import Annotated
from typing_extensions import TypedDict


from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [15]:
from typing import Annotated
from langchain_tavily import  TavilySearch
from langchain_core.tools import tool 
from typing_extensions import TypedDict


from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph ,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition

from langgraph.types import Command,interrupt 

class State(TypedDict):
    
    messages: Annotated[list,add_messages]

graph_builder =StateGraph(State)

@tool
def human_assistance(query:str)->str:
    """Request assistance from a human"""
    human_response =interrupt ({"query": query})
    return human_response["data"]

tool = TavilySearch(max_results=2)
tools = [tool, human_assistance]

llm_with_tools = llm.bind_tools(tools)
def chatbot(state:State):
    message = llm_with_tools.invoke(state["messages"])
    # Because we will be interrupting during tool excuetion,
    # we disable parallel tool calling to avoid repeating any 
    # tool invocation when we resume .

    return {"messages":[message]}

graph_builder.add_node("chatbot",chatbot )
tool_node= ToolNode(tools=tools)
graph_builder.add_node("tools",tool_node)
graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

In [16]:
from IPython.display import Image,display
try:
    memory=MemorySaver()
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional 
    pass

In [17]:
user_input ="I need some expert guidance and assistance for building an Ai agent .could you  request assistance for me ?"

config={"configurable":{"thread_id":"1"}}

events = graph.stream(
    {"messages": user_input},
    config,
    stream_mode ="values",
)
for event in events:
    if "messages" in event :
        event["messages"][-1].pretty_print()

NameError: name 'graph' is not defined

In [ ]:
human_response =( 
    "we , the experts are here to help ! We had recommended you check out langgraph to build your agent."
    "it is much more reliable and extensible than simple autonoumous agents. "
)

human_command =Command(resume={"data": human_response})

events = graph.stream(human_commmand,config,stream_mode="values")
for events in events:
    if "messages " in event :
        event["messages"][-1].preety_print()